In [2]:
# ────────────────────────────── ① 依赖 & 自家模块 ──────────────────────────────
import numpy as np
import pandas as pd
import datetime, pathlib, sys

# 把当前目录加入 sys.path 以便 import 本地 .py
sys.path.insert(0, str(pathlib.Path.cwd()))

from stateprep       import single_vortex
from spec_classic    import SpectralSolver
from rk4_realspace   import RK4Solver
from qspec_naive     import QuantumSpectralSolver
from qspec_pod       import QuantumPODSolver
import metrics as mt                              # 前一步给出的 metrics.py

# ────────────────────────────── ② 常量：时间网格 & 参数表 ───────────────────────
T_GRID = np.linspace(0, 0.3, 4)               # 6 个快照
Δt     = T_GRID[1] - T_GRID[0]                    # 0.314...
N_snap = len(T_GRID)

PARAM_LIST = [
    (-0.39,  1.42, 0.18), (-0.39,  1.42, 0.25), (-0.39,  1.42, 0.54),
    (-1.08, -1.08, 0.18), (-1.08, -1.08, 0.25), (-1.08, -1.08, 0.54),
    (-1.39,  1.15, 0.18), (-1.39,  1.15, 0.25), (-1.39,  1.15, 0.54),
    ( 0.73,  0.31, 0.18), ( 0.73,  0.31, 0.25), ( 0.73,  0.31, 0.54),
]

# ────────────────────────────── ③ 指标函数按顺序收集 ──────────────────────────
METRICS = [
    ("density_mse",        mt.compute_density_mse),
    ("fidelity",           mt.compute_fidelity),
    ("l2_error",           mt.compute_relative_error),
    ("spectrum_l2",        mt.compute_spectrum_l2_error),
]


# ────────────────────────────── ④ 参与评估的算法实例 ──────────────────────────
METHODS = {
    "RK4"        : RK4Solver(),
    "Quantum"    : QuantumSpectralSolver(),
    "QuantumPOD" : QuantumPODSolver(),   # 未实现时自动跳过
}

# ────────────────────────────── ⑤ 主循环：计算并写 Excel ───────────────────────
# 生成文件名含 Δt 与快照数
ts  = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
fname = f"metrics_dt{Δt:.2f}_n{N_snap}_{ts}.xlsx"
writer = pd.ExcelWriter(fname, engine="openpyxl")

for meth, solver in METHODS.items():
    rows = []
    for (x0, y0, σ) in PARAM_LIST:
        tag = f"x{x0:+.2f}_y{y0:+.2f}_s{σ:.2f}"

        # baseline
        ψ1_0, ψ2_0 = single_vortex(σ, (x0, y0))
        classic = SpectralSolver().evolve(ψ1_0, ψ2_0, T_GRID)

        # peer
        try:
            peer = solver.evolve(ψ1_0, ψ2_0, T_GRID)
        except NotImplementedError:
            print(f"[{meth}] 未实现，跳过 {tag}")
            continue

        # 四行指标 + 空行
        first = True
        for name, fn in METRICS:
            vals = [fn(*peer[t], *classic[t]) for t in T_GRID]
            rows.append([tag if first else "", name, *vals])
            first = False
        rows.append([""] * (2 + N_snap))          # 空行分块

    # 写入 sheet
    col_names = ["param", "metric"] + [f"t={t:.2f}" for t in T_GRID]
    pd.DataFrame(rows, columns=col_names).to_excel(writer, sheet_name=meth, index=False)
    print(f"✓ {meth} 完成")

writer.close()
print("★ 已保存 →", fname)


✓ RK4 完成
✓ Quantum 完成
✓ QuantumPOD 完成
★ 已保存 → metrics_dt0.10_n4_20250802_160548.xlsx
